Data Preprocessing

In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.feature_selection import mutual_info_classif
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split


In [ ]:
# Reading in the datasets
additional_test = pd.read_csv('test/Features/additional_features.csv')
color_test = pd.read_csv('test/Features/color_histogram.csv')
hog_test = pd.read_csv('test/Features/hog_pca.csv')
testing = pd.read_csv('test/test_metadata.csv')

additional_train = pd.read_csv('train/Features/additional_features.csv')
color_train = pd.read_csv('train/Features/color_histogram.csv')
hog_train = pd.read_csv('train/Features/hog_pca.csv')
training = pd.read_csv('train/train_metadata.csv')


# PREPROCESSING THE DATA

# Standardising additional_features 
additional_features_columns = ['edge_density', 'mean_b', 'mean_g', 'mean_r']
scaler_additional = StandardScaler()
additional_train[additional_features_columns] = scaler_additional.fit_transform(additional_train[additional_features_columns])
additional_test[additional_features_columns] = scaler_additional.transform(additional_test[additional_features_columns])

# Standardising color histograms
color_columns = [col for col in color_train.columns if col.startswith("ch_")]
scaler_color = StandardScaler()
color_train[color_columns] = scaler_color.fit_transform(color_train[color_columns])
color_test[color_columns] = scaler_color.transform(color_test[color_columns])

# Once preprocessed, we can now merge the features together for the training metadata set
combined_training = pd.merge(training, additional_train, on='image_path')
combined_training = pd.merge(combined_training, color_train, on='image_path')
combined_training = pd.merge(combined_training, hog_train, on='image_path')
combined_training = combined_training[[col for col in combined_training.columns if col != 'ClassId'] + ['ClassId']]

# Same for the testing metadata set
combined_testing = pd.merge(testing, additional_test, on='image_path')
combined_testing = pd.merge(combined_testing, color_test, on='image_path')  
combined_testing = pd.merge(combined_testing, hog_test, on='image_path')
combined_testing = combined_testing[[col for col in combined_testing.columns if col != 'ClassId'] + ['ClassId']]


# FEATURE SELECTION via Mutual Information
def apply_mutual_info_filtering(combined_training, top_k=50):
    X = combined_training.drop(columns=['id', 'image_path', 'ClassId'])
    y = combined_training['ClassId']
    
    mi_scores = mutual_info_classif(X, y, discrete_features=False, random_state=42)
    mi_df = pd.DataFrame({'feature': X.columns, 'mutual_info': mi_scores})
    mi_df_sorted = mi_df.sort_values(by='mutual_info', ascending=False)
    top_features = mi_df_sorted.head(top_k)['feature'].tolist()
    
    return combined_training[top_features + ['ClassId']], mi_df_sorted


print("Applying Mutual Information Filtering to Training Data:")
training_filtered, mi_df_train_sorted = apply_mutual_info_filtering(combined_training, top_k=50)
print("Top 50 Features based on Mutual Information:")

testing_filtered = combined_testing[training_filtered.columns.tolist()]


Applying Mutual Information Filtering to Training Data:
Top 50 Features based on Mutual Information:
      hog_pca_0  hog_pca_3  hog_pca_1  hog_pca_2  hog_pca_5  hog_pca_4  \
0      0.597466  -1.971381   1.520531   0.411531  -1.018461  -0.594053   
1     -2.208741  -0.781442  -0.824214  -0.535469  -0.413522   0.948866   
2     -0.001004   0.505151   0.975770  -0.058772   0.021935   0.527402   
3     -0.811440   0.743867   0.536341   1.365921   0.049751   0.279398   
4     -2.171802   0.492189  -0.642769   0.826297  -0.697473  -0.146542   
...         ...        ...        ...        ...        ...        ...   
2348   2.015280  -0.112696  -0.321264  -0.063573   1.130592   0.399974   
2349  -1.646095  -0.649299   0.210245   1.087668  -0.270195   0.713822   
2350   3.703216   0.059137  -1.017537   0.077843  -1.380024   0.681139   
2351  -1.248648  -0.075060  -0.723521  -1.700338  -0.135474   0.313651   
2352   0.323162  -0.362392   0.561640   0.150786   0.192753  -0.131690   

      hog_

Benchmark Model

In [117]:
from sklearn.dummy import DummyClassifier

X_train = np.array(training_filtered.drop(columns=['ClassId']))
Y_train = np.array(training_filtered['ClassId'])

X_train_split, X_val, Y_train_split, Y_val = train_test_split(X_train, Y_train, test_size=0.3, random_state=42)


dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, Y_train)
y_dummy = dummy.predict(X_val)
print("Baseline Accuracy:", accuracy_score(Y_val, y_dummy))

Baseline Accuracy: 0.058287795992714025


Neural Network Model

In [130]:
# NEURAL NETWORK
X_train = np.array(training_filtered.drop(columns=['ClassId']))
Y_train = np.array(training_filtered['ClassId'])

m, n = X_train.shape

shuffled_df = training_filtered.sample(frac=1, random_state=42).reset_index(drop=True)
X_train = shuffled_df.drop(columns=['ClassId']).to_numpy()
Y_train = shuffled_df['ClassId'].to_numpy()
X = X_train.T

numInputs = X.shape[0]
numHidden = 256
numOutput = 43


# Probably a good idea to integrate some form of dropout here
# Necessary functions for the neural network
def init_params():
    W1 = np.random.randn(numHidden, numInputs) * np.sqrt(2. / numInputs)
    b1 = np.zeros((numHidden, 1))
    W2 = np.random.randn(numOutput, numHidden) * np.sqrt(2. / numHidden)
    b2 = np.zeros((numOutput, 1))
    return W1, b1, W2, b2

def forward_propagation(X, W1, b1, W2, b2, dropout_rate=0.5, training=True):
    Z1 = np.dot(W1, X) + b1
    A1 = np.maximum(0, Z1)  # ReLU

    if training:
        # Dropout mask: 1 with probability (1 - dropout_rate), 0 otherwise
        dropout_mask = (np.random.rand(*A1.shape) > dropout_rate).astype(float)
        A1 *= dropout_mask  # Apply mask
        A1 /= (1 - dropout_rate)  # Inverted dropout scaling

    Z2 = np.dot(W2, A1) + b2
    exp_scores = np.exp(Z2 - np.max(Z2, axis=0, keepdims=True))
    A2 = exp_scores / np.sum(exp_scores, axis=0, keepdims=True)
    return A1, A2


def backward_propagation(X, Y, A1, A2, W1, W2):
    m = X.shape[1]
    dZ2 = A2 - Y 
    dW2 = (1 / m) * np.dot(dZ2, A1.T)
    db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)
    
    dZ1 = np.dot(W2.T, dZ2) * (A1 > 0)
    dW1 = (1 / m) * np.dot(dZ1, X.T)
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)
    
    return dW1, db1, dW2, db2

def one_hot_encode(Y, num_classes):
    m = Y.shape[0]
    Y_encoded = np.zeros((num_classes, m))
    Y_encoded[Y, np.arange(m)] = 1
    return Y_encoded

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate):
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    return W1, b1, W2, b2

def get_accuracy(A2, Y):
    predictions = np.argmax(A2, axis=0)
    accuracy = np.mean(predictions == Y)
    return accuracy*100

def predict(A2):
    return np.argmax(A2, axis=0)

def gradient_descent(X, Y, W1, b1, W2, b2, learning_rate=0.01, epochs=1000, dropout_rate=0.5):
    Y_encoded = one_hot_encode(Y, numOutput)
    for epoch in range(epochs):
        # Apply dropout only during training
        A1, A2 = forward_propagation(X, W1, b1, W2, b2, dropout_rate=dropout_rate, training=True)

        dW1, db1, dW2, db2 = backward_propagation(X, Y_encoded, A1, A2, W1, W2)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)

        if epoch % 100 == 0:
            A1_val, A2_val = forward_propagation(X, W1, b1, W2, b2, training=False)
            log_probs = -np.log(A2_val[Y, np.arange(Y.shape[0])])
            loss = np.mean(log_probs)
            acc = get_accuracy(A2_val, Y)
            print(f"Epoch {epoch}, Loss: {loss:.4f}, Accuracy: {acc:.2f}%")

    return W1, b1, W2, b2


X = X_train.T 
Y = Y_train.astype(int)

W1, b1, W2, b2 = gradient_descent(X, Y, *init_params(), learning_rate=0.1, epochs=1000)


# Confusion Matrix

'''
A1, A2 = forward_propagation(X, W1, b1, W2, b2)
Y_pred = predict(A2)
conf_matrix = confusion_matrix(Y_train, Y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=False, cmap='Blues', fmt='g')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(Y_train, Y_pred))
'''



Epoch 0, Loss: 4.2967, Accuracy: 3.15%
Epoch 100, Loss: 1.8033, Accuracy: 51.29%
Epoch 200, Loss: 1.4587, Accuracy: 58.98%
Epoch 300, Loss: 1.2825, Accuracy: 63.76%
Epoch 400, Loss: 1.1670, Accuracy: 67.04%
Epoch 500, Loss: 1.0824, Accuracy: 69.42%
Epoch 600, Loss: 1.0155, Accuracy: 71.68%
Epoch 700, Loss: 0.9602, Accuracy: 73.27%
Epoch 800, Loss: 0.9143, Accuracy: 74.56%
Epoch 900, Loss: 0.8739, Accuracy: 75.84%


'\nA1, A2 = forward_propagation(X, W1, b1, W2, b2)\nY_pred = predict(A2)\nconf_matrix = confusion_matrix(Y_train, Y_pred)\nplt.figure(figsize=(10, 8))\nsns.heatmap(conf_matrix, annot=False, cmap=\'Blues\', fmt=\'g\')\nplt.xlabel("Predicted")\nplt.ylabel("True")\nplt.title("Confusion Matrix")\nplt.show()\n\nprint(classification_report(Y_train, Y_pred))\n'

In [ ]:
# Predicting on the test set
X_test = np.array(testing_filtered.drop(columns=['ClassId']))
X_test_np = X_test.T

_, A2_test = forward_propagation(X_test_np, W1, b1, W2, b2)
predicted_classes = np.argmax(A2_test, axis=0)

NN_output_df = pd.DataFrame({
    'id': combined_testing['id'].values,
    'ClassId': predicted_classes
})

NN_output_df.to_csv('submission.csv', index=False)


Naive Bayes Classification

In [ ]:
X_train_split, X_val, Y_train_split, Y_val = train_test_split(X_train, Y_train, test_size=0.3, random_state=42)

nb_model = GaussianNB()
nb_model.fit(X_train_split, Y_train_split)

y_pred_nb = nb_model.predict(X_val)

# Evaluating
print("Naive Bayes Accuracy:", accuracy_score(Y_val, y_pred_nb))
print(classification_report(Y_val, y_pred_nb))

Naive Bayes Accuracy: 0.33697632058287796
              precision    recall  f1-score   support

           0       0.09      0.25      0.13         8
           1       0.37      0.40      0.38        85
           2       0.29      0.25      0.27        93
           3       0.36      0.20      0.26        50
           4       0.47      0.08      0.13        91
           5       0.29      0.09      0.14        79
           6       0.03      1.00      0.05        14
           7       0.31      0.16      0.21        68
           8       0.09      0.07      0.08        59
           9       0.55      0.48      0.51        54
          10       0.39      0.11      0.17        85
          11       0.33      0.13      0.19        53
          12       1.00      0.84      0.91        89
          13       1.00      0.73      0.84       104
          14       0.83      0.79      0.81        24
          15       0.50      0.35      0.42        31
          16       0.50      0.35      

Ensemble Stacking Model

In [92]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

X_train = np.array(training_filtered.drop(columns=['ClassId']))
Y_train = np.array(training_filtered['ClassId'])

X_train_split, X_val, Y_train_split, Y_val = train_test_split(X_train, Y_train, test_size=0.3, random_state=42)

# Level-0 base models
rf = RandomForestClassifier(random_state=1)
lr = LogisticRegression(max_iter=1000)
nb = GaussianNB()

# Train base models on training data
rf.fit(X_train, Y_train)
lr.fit(X_train, Y_train)
nb.fit(X_train, Y_train)

# Predict on validation set
rf_pred = rf.predict(X_val)
lr_pred = lr.predict(X_val)
nb_pred = nb.predict(X_val)

# Stack predictions as features for meta-model
meta_X = np.vstack((rf_pred, lr_pred, nb_pred)).T
meta_y = Y_val

# Level-1 meta-model
meta_model = LogisticRegression()
meta_model.fit(meta_X, meta_y)

# Evaluate the stacked model
meta_val_pred = meta_model.predict(meta_X)
print("Stacked Model Accuracy:", accuracy_score(meta_y, meta_val_pred))


# While uncertain about the performance of this model, we can assume it sucks because of the low accuracy of the base models, namely the Naive Bayes model.

Stacked Model Accuracy: 0.3381906496660595


C:\Users\jason\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
